In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score
from sklearn import tree
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier


**Read Loaded Data**

In [ ]:
def load_and_preprocess_data():
    # Load data
    data = pd.read_csv(r"magic04.data")
    print("Data loaded successfully:\n", data)

    # Set column names
    data.columns = ['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'class']
    ("Data \n", data)

    # Handle missing values (if any)
    data.dropna(inplace=True)

    # Normalizing features between 0 and 1
    scaler = MinMaxScaler()
    data[['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10']] = scaler.fit_transform(
    data[['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10']])
    print("Data after scaling:\n", data)

  
    gamma_data = data[data['class'] == 'g']  # 12,331
    hadron_data = data[data['class'] == 'h']  # 6,688

    
    gamma_data_equal = gamma_data.sample(n=len(hadron_data), random_state=42)
    #concatination
    balanced_data = pd.concat([gamma_data_equal, hadron_data])
    print("Data after balancing: \n", balanced_data)  # 13,376 (6,688+6,688)
 
    #Shuffle the combined dataset to ensure randomness 
    balanced_data = balanced_data.sample(frac=1, random_state=42).reset_index(drop=True)
    
    #Split the dataset into training 70% and testing sets 30%
    train_data, test_data = train_test_split(balanced_data, test_size=0.3, random_state=42)
    print("Testing: ", len(test_data))    # 13,376*30% = 4012.8 = 4013
    print("Training: ", len(train_data))  # 13,376*70% = 9,363.2 = 9363

    X_train = train_data.drop(columns=['class']) #10 features
    y_train = train_data['class']                #target class [g,h]
    print("Features of Training Set: ", len(X_train))
    print("Label of Training Set: ", len(y_train))

    X_test = test_data.drop(columns=['class'])
    y_test = test_data['class']

    return X_train, y_train, X_test, y_test

In [3]:
load_and_preprocess_data()

Data loaded successfully:
         28.7967   16.0021  2.6449  0.3918  0.1982   27.7004    22.011  \
0       31.6036   11.7235  2.5185  0.5303  0.3773   26.2722   23.8238   
1      162.0520  136.0310  4.0612  0.0374  0.0187  116.7410  -64.8580   
2       23.8172    9.5728  2.3385  0.6147  0.3922   27.2107   -6.4633   
3       75.1362   30.9205  3.1611  0.3168  0.1832   -5.5277   28.5525   
4       51.6240   21.1502  2.9085  0.2420  0.1340   50.8761   43.1887   
...         ...       ...     ...     ...     ...       ...       ...   
19014   21.3846   10.9170  2.6161  0.5857  0.3934   15.2618   11.5245   
19015   28.9452    6.7020  2.2672  0.5351  0.2784   37.0816   13.1853   
19016   75.4455   47.5305  3.4483  0.1417  0.0549   -9.3561   41.0562   
19017  120.5135   76.9018  3.9939  0.0944  0.0683    5.8043  -93.5224   
19018  187.1814   53.0014  3.2093  0.2876  0.1539 -167.3125 -168.4558   

       -8.2027   40.092   81.8828  g  
0      -9.9574   6.3609  205.2610  g  
1     -45.2160  76

(             f1        f2        f3        f4        f5        f6        f7  \
 6552   0.151152  0.045198  0.170284  0.371974  0.272040  0.453564  0.659253   
 5163   0.094367  0.041728  0.171053  0.496761  0.351015  0.472952  0.614485   
 2889   0.057175  0.055865  0.190124  0.553358  0.410431  0.432480  0.565816   
 5412   0.204553  0.098790  0.258989  0.340039  0.260187  0.508342  0.686033   
 12705  0.049029  0.040989  0.083205  0.665644  0.509705  0.437764  0.611302   
 ...         ...       ...       ...       ...       ...       ...       ...   
 11964  0.057456  0.052304  0.208043  0.525742  0.447622  0.440127  0.554566   
 5191   0.083938  0.052384  0.291248  0.468576  0.325233  0.456398  0.616922   
 5390   0.073981  0.075989  0.204820  0.304580  0.211291  0.470812  0.557314   
 860    0.070069  0.039636  0.186753  0.595863  0.461105  0.474066  0.615844   
 7270   0.061749  0.057256  0.182910  0.498579  0.368944  0.452380  0.613293   
 
              f8        f9       f10  

**Naive Bayes Classifier**

In [4]:
def naive_bayes():
    # Load and preprocess data
    X_train, y_train, X_test, y_test = load_and_preprocess_data()
    
    #fit on training set using features and label
    '''
    The Gaussian Naive Bayes algorithm is a probabilistic classification algorithm
    that is based on Bayes' theorem with the assumption of independence between 
    features (as this we call the term "naive"). It is often used for classification tasks
    when the features are continuous and assumed to have a Gaussian (normal) distribution.
    '''
    nb = GaussianNB()
    nb.fit(X_train, y_train) 
    
    #Cross Validation to evaluate the performance
    '''
     Cross Validation to evaluate the performance
    Step-by-step explanation of how the data is split into 5 folds:
        a)The entire dataset is divided into 5 equal-sized subsets.
        b)In the first iteration, the first fold is used as the validation set, and the remaining 4 folds are used for training.
        c)In the second iteration, the second fold is used as the validation set, and the other 4 folds are used for training.
        d)This process continues for the remaining folds until all 5 folds have been used as the validation set once.
        e)The performance of the model is evaluated on each fold, and the results are typically averaged to obtain an overall performance measure.
    '''
    nb_cv = cross_val_score(nb, X_train, y_train, cv=5) # cv=5 specifies 5 fold cross validation
    print("Naive Bayes Cross-Validation Accuracy: {:.4f}%".format(nb_cv.mean() * 100))
    
    '''
    Reporting Naive Bias accuracy, precision, recall, F-score & confusion matrix
    '''
    from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
    #predict labels of testing features
    nb_predictions = nb.predict(X_test) 
    
    #Accuracy on testing features and label
    accuracy = accuracy_score(y_test, nb_predictions)
    print("Naive Bayes Testing Accuracy: {:.4f}%".format(accuracy * 100))
    
    #Calculate confusion matrix
    nb_conf_matrix = confusion_matrix(y_test, nb_predictions)
    print("Naive Bayes Confusion Matrix:")
    print(nb_conf_matrix)
    
    #Calculate precision, recall and F1-score 
    nb_report = classification_report(y_test, nb_predictions)
    print("Naive Bayes Classification Report:")
    print(nb_report)


In [5]:
naive_bayes()

Data loaded successfully:
         28.7967   16.0021  2.6449  0.3918  0.1982   27.7004    22.011  \
0       31.6036   11.7235  2.5185  0.5303  0.3773   26.2722   23.8238   
1      162.0520  136.0310  4.0612  0.0374  0.0187  116.7410  -64.8580   
2       23.8172    9.5728  2.3385  0.6147  0.3922   27.2107   -6.4633   
3       75.1362   30.9205  3.1611  0.3168  0.1832   -5.5277   28.5525   
4       51.6240   21.1502  2.9085  0.2420  0.1340   50.8761   43.1887   
...         ...       ...     ...     ...     ...       ...       ...   
19014   21.3846   10.9170  2.6161  0.5857  0.3934   15.2618   11.5245   
19015   28.9452    6.7020  2.2672  0.5351  0.2784   37.0816   13.1853   
19016   75.4455   47.5305  3.4483  0.1417  0.0549   -9.3561   41.0562   
19017  120.5135   76.9018  3.9939  0.0944  0.0683    5.8043  -93.5224   
19018  187.1814   53.0014  3.2093  0.2876  0.1539 -167.3125 -168.4558   

       -8.2027   40.092   81.8828  g  
0      -9.9574   6.3609  205.2610  g  
1     -45.2160  76

**Decision Tree Classifier**

In [6]:
def decision_tree():
    
    X_train, y_train, X_test, y_test = load_and_preprocess_data()


    dt = tree.DecisionTreeClassifier().fit(X_train , y_train)
    
    '''
     Cross Validation to evaluate the performance
    Step-by-step explanation of how the data is split into 5 folds:
        a)The entire dataset is divided into 5 equal-sized subsets.
        b)In the first iteration, the first fold is used as the validation set, and the remaining 4 folds are used for training.
        c)In the second iteration, the second fold is used as the validation set, and the other 4 folds are used for training.
        d)This process continues for the remaining folds until all 5 folds have been used as the validation set once.
        e)The performance of the model is evaluated on each fold, and the results are typically averaged to obtain an overall performance measure.
    '''
    dt_cv = cross_val_score(dt, X_train, y_train, cv=5)  # cv=5 specifies 5 fold cross validation
    print("Decision Tree Cross-Validation Accuracy: {:.4f}%".format(dt_cv.mean() * 100))

    '''
    Reporting Decision Tree accuracy, precision, recall, F-score & confusion matrix
    '''
    
    dt_predictions = dt.predict(X_test)  # Predict labels of testing features

    # Accuracy on testing features and label
    accuracy = accuracy_score(y_test, dt_predictions)
    print("Decision Tree Testing Accuracy: {:.4f}%".format(accuracy * 100))

    # Calculate confusion matrix
    dt_conf_matrix = confusion_matrix(y_test, dt_predictions)
    print("Decision Tree Confusion Matrix:")
    print(dt_conf_matrix)

    # Calculate precision, recall and F1-score
    dt_report = classification_report(y_test, dt_predictions)
    print("Decision Tree Classification Report:")
    print(dt_report)


In [7]:
decision_tree()

Data loaded successfully:
         28.7967   16.0021  2.6449  0.3918  0.1982   27.7004    22.011  \
0       31.6036   11.7235  2.5185  0.5303  0.3773   26.2722   23.8238   
1      162.0520  136.0310  4.0612  0.0374  0.0187  116.7410  -64.8580   
2       23.8172    9.5728  2.3385  0.6147  0.3922   27.2107   -6.4633   
3       75.1362   30.9205  3.1611  0.3168  0.1832   -5.5277   28.5525   
4       51.6240   21.1502  2.9085  0.2420  0.1340   50.8761   43.1887   
...         ...       ...     ...     ...     ...       ...       ...   
19014   21.3846   10.9170  2.6161  0.5857  0.3934   15.2618   11.5245   
19015   28.9452    6.7020  2.2672  0.5351  0.2784   37.0816   13.1853   
19016   75.4455   47.5305  3.4483  0.1417  0.0549   -9.3561   41.0562   
19017  120.5135   76.9018  3.9939  0.0944  0.0683    5.8043  -93.5224   
19018  187.1814   53.0014  3.2093  0.2876  0.1539 -167.3125 -168.4558   

       -8.2027   40.092   81.8828  g  
0      -9.9574   6.3609  205.2610  g  
1     -45.2160  76

**AdaBoost Classifier**

In [8]:
def adaboost():
    # Load and preprocess data
    X_train, y_train, X_test, y_test = load_and_preprocess_data()
    
    # AdaBoost classifier
    '''
    AdaBoost (Adaptive Boosting) is a machine learning ensemble method that combines 
    multiple weak classifiers to create a strong classifier. It is a boosting algorithm 
    that iteratively trains weak classifiers on different subsets of the training data 
    and assigns weights to each instance based on their classification performance. 
    The final prediction is made by combining the predictions of these weak classifiers.
    
    AdaBoost uses decision trees with a single split (decision stumps) as weak learners.
    '''
    adaboost = AdaBoostClassifier()
    
    # Cross Validation to evaluate the performance
    '''
    Here's how the data is divided in each iteration of the 5-fold cross-validation:
        
    Iteration 1:
    Testing set: Fold 1
    Training set: Folds 2, 3, 4, 5
    
    Iteration 2:
    Testing set: Fold 2
    Training set: Folds 1, 3, 4, 5
    
    Iteration 3:
    Testing set: Fold 3
    Training set: Folds 1, 2, 4, 5
    
    Iteration 4:
    Testing set: Fold 4
    Training set: Folds 1, 2, 3, 5
    
    Iteration 5:
    Testing set: Fold 5
    Training set: Folds 1, 2, 3, 4
    '''
                              # ckassifier train_features  train_target
    adaboost_cv = cross_val_score(adaboost, X_train, y_train, cv=5)
    print("AdaBoost Cross-Validation Accuracy: {:.4f}%".format(adaboost_cv.mean() * 100))
    
    # Fit on training set using features and label
    adaboost.fit(X_train, y_train)
    
    # Predict labels of testing features
    adaboost_predictions = adaboost.predict(X_test)
    
    # Accuracy on testing features and label
    '''
    "accuracy_score" is used to calculate the accuracy on a specific set of
    predicted and true labels, 
    while "adaboost_cv.mean()" calculates the average accuracy across multiple
    folds in the cross-validation process. The choice of which one to use depends
    on the specific evaluation needs and the nature of the data.
    '''
    accuracy = accuracy_score(y_test, adaboost_predictions)
    print("AdaBoost Testing Accuracy: {:.4f}%".format(accuracy * 100))
    
    # Calculate confusion matrix
    '''
      r
    p[[TP(g) FP       => acc= (tp+tn)/(tp+tn+fp+fn) =>specifity= tn/(tn+fp)
      FN    TN(h)]]  =>precision= tp/(tp+fp)       =>recalll= tp/(tp+fn)
              s
    '''
    adaboost_conf_matrix = confusion_matrix(y_test, adaboost_predictions)
    print("AdaBoost Confusion Matrix:")
    print(adaboost_conf_matrix)
    
    print("\n")
    
    # Calculate precision, recall, and F1-score
    adaboost_report = classification_report(y_test, adaboost_predictions)
    print("AdaBoost Classification Report:")
    print(adaboost_report)
    
    print("Wait Tuning....\n")
    
    # Model Parameter Tuning
    # Define the parameter grid for AdaBoost
    '''
    the hyperparameter being tuned is 'n_estimators', which represents the number of
    estimators (weak learners) in the AdaBoost ensemble. Each estimator contributes 
    to the final prediction, and a higher number of estimators can potentially 
    improve the model's performance.
    '''
    param_grid = {'n_estimators': [50, 100, 200]}
    
    # Perform grid search with cross-validation
    '''
    perform a grid search for hyperparameter tuning using the AdaBoostClassifier model
    and the specified parameter grid. It systematically explores different combinations
    of hyperparameters, trains and evaluates the model using cross-validation, 
    and stores the best-performing combination of hyperparameters for later use.
    '''
    #perform 15 iterations.
    grid_search = GridSearchCV(AdaBoostClassifier(), param_grid, cv=5)
    #To see results: cv_results = grid_search.cv_results_
    grid_search.fit(X_train, y_train)
    
    # Get the best classifier with tuned parameters
    best_adaboost = grid_search.best_estimator_
    
    # Predict the labels for the testing data using the best classifier
    best_adaboost_predictions = best_adaboost.predict(X_test)
    
    # Calculate accuracy on testing data using the best classifier
    accuracy_tuned = accuracy_score(y_test, best_adaboost_predictions)
    print("Tuned AdaBoost Testing Accuracy: {:.4f}%".format(accuracy_tuned * 100))
    
    # Calculate confusion matrix using the best classifier
    confusion_tuned = confusion_matrix(y_test, best_adaboost_predictions)
    print("Tuned AdaBoost Confusion Matrix:")
    print(confusion_tuned)
    
    print("\n")
    
    # Calculate precision, recall, and F1-score using the best classifier
    report_tuned = classification_report(y_test, best_adaboost_predictions)
    print("Tuned AdaBoost Classification Report:")
    print(report_tuned)
    
    '''
1. **Bagging (Bootstrap Aggregating)**:
   - Bagging involves training multiple instances of the same base learning algorithm on different subsets of the training data.
   - Each subset of the training data is sampled with replacement (bootstrap sampling), resulting in multiple bootstrap samples.
   - The base learning algorithm is trained independently on each bootstrap sample, resulting in multiple base models.
   - During prediction, the final prediction is typically obtained by averaging the predictions of all base models (for regression) 
       or by taking a majority vote (for classification).
   - Examples of bagging algorithms include Random Forest, which uses decision trees as base models, and Bagged Decision Trees.

2. **Boosting**:
   - Boosting involves sequentially training multiple weak learners (base models) and adjusting the weights of training instances based on
       the performance of previously trained models.
   - In each iteration, the algorithm gives more weight to instances that were misclassified or poorly predicted by previous models, 
       thereby focusing on "hard" instances.
   - The weak learners are typically simple models, such as decision trees with limited depth (e.g., decision stumps).
   - The final prediction is a weighted sum of the predictions of all weak learners, where the weights are determined during training based on
       the performance of each weak learner.
   - Examples of boosting algorithms include AdaBoost, Gradient Boosting Machines (GBM), and XGBoost.

**Differences**:

1. **Approach to Combining Models**:
   - In bagging, multiple base models are trained independently on different subsets of the data, and their predictions are combined by averaging or voting.
   - In boosting, base models are trained sequentially, and each subsequent model focuses more on instances that were misclassified by previous models.

2. **Weighting of Training Instances**:
   - Bagging assigns equal weight to all training instances in each bootstrap sample.
   - Boosting assigns higher weights to instances that were misclassified or poorly predicted by previous models, thereby focusing more on difficult instances.

3. **Handling of Overfitting**:
   - Bagging reduces overfitting by reducing the variance of the model through averaging predictions from multiple base models.
   - Boosting reduces bias and variance by iteratively improving the model's ability to fit the training data, often resulting in better performance than
       individual weak learners.

In summary, while bagging and boosting are both ensemble learning techniques that combine multiple models,
            they differ in their approach to constructing the ensemble, handling of training instances, and addressing the problem of overfitting.
            Bagging focuses on reducing variance by averaging predictions from independently trained models, while boosting focuses on iteratively improving
            the model's performance by emphasizing difficult instances during training.
    '''
    

In [9]:
adaboost()

Data loaded successfully:
         28.7967   16.0021  2.6449  0.3918  0.1982   27.7004    22.011  \
0       31.6036   11.7235  2.5185  0.5303  0.3773   26.2722   23.8238   
1      162.0520  136.0310  4.0612  0.0374  0.0187  116.7410  -64.8580   
2       23.8172    9.5728  2.3385  0.6147  0.3922   27.2107   -6.4633   
3       75.1362   30.9205  3.1611  0.3168  0.1832   -5.5277   28.5525   
4       51.6240   21.1502  2.9085  0.2420  0.1340   50.8761   43.1887   
...         ...       ...     ...     ...     ...       ...       ...   
19014   21.3846   10.9170  2.6161  0.5857  0.3934   15.2618   11.5245   
19015   28.9452    6.7020  2.2672  0.5351  0.2784   37.0816   13.1853   
19016   75.4455   47.5305  3.4483  0.1417  0.0549   -9.3561   41.0562   
19017  120.5135   76.9018  3.9939  0.0944  0.0683    5.8043  -93.5224   
19018  187.1814   53.0014  3.2093  0.2876  0.1539 -167.3125 -168.4558   

       -8.2027   40.092   81.8828  g  
0      -9.9574   6.3609  205.2610  g  
1     -45.2160  76

c:\Users\LENOVO\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\LENOVO\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\LENOVO\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\LENOVO\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users

AdaBoost Cross-Validation Accuracy: 80.9034%


c:\Users\LENOVO\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


AdaBoost Testing Accuracy: 80.3887%
AdaBoost Confusion Matrix:
[[1608  368]
 [ 419 1618]]


AdaBoost Classification Report:
              precision    recall  f1-score   support

           g       0.79      0.81      0.80      1976
           h       0.81      0.79      0.80      2037

    accuracy                           0.80      4013
   macro avg       0.80      0.80      0.80      4013
weighted avg       0.80      0.80      0.80      4013

Wait Tuning....



c:\Users\LENOVO\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\LENOVO\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\LENOVO\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\LENOVO\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users

Tuned AdaBoost Testing Accuracy: 81.8590%
Tuned AdaBoost Confusion Matrix:
[[1644  332]
 [ 396 1641]]


Tuned AdaBoost Classification Report:
              precision    recall  f1-score   support

           g       0.81      0.83      0.82      1976
           h       0.83      0.81      0.82      2037

    accuracy                           0.82      4013
   macro avg       0.82      0.82      0.82      4013
weighted avg       0.82      0.82      0.82      4013



**Random Forests Classifier**

In [10]:
def random_forest():
    # Load and preprocess data
    X_train, y_train, X_test, y_test = load_and_preprocess_data()
    
    # Random Forest classifier
    random_forest = RandomForestClassifier()
    
    # Cross Validation to evaluate the performance
    '''
     Here's how the data is divided in each iteration of the 5-fold cross-validation:

     Iteration 1:
     Testing set: Fold 1
     Training set: Folds 2, 3, 4, 5
     
     Iteration 2:
     Testing set: Fold 2
     Training set: Folds 1, 3, 4, 5
     
     Iteration 3:
     Testing set: Fold 3
     Training set: Folds 1, 2, 4, 5
     
     Iteration 4:
     Testing set: Fold 4
     Training set: Folds 1, 2, 3, 5
     
     Iteration 5:
     Testing set: Fold 5
     Training set: Folds 1, 2, 3, 4
    '''
                                  # ckassifier train_features  train_target
    random_forest_cv = cross_val_score(random_forest, X_train, y_train, cv=5)
    #print("Random Forest Cross-Validation Accuracy: {:.4f}%".format(random_forest_cv.mean() * 100))
    print("Random Forest Cross-Validation Accuracy: {:.4f}%".format(random_forest_cv.mean() * 100))
    
    # Fit on training set using features and label
    '''
    Algorithm looks for the best splits in the data to minimize impurity (or entropy)
    and maximize information gain(measure of impurity - Entropy).
    '''
    random_forest.fit(X_train, y_train)
    
    # Predict labels of testing features
    random_forest_predictions = random_forest.predict(X_test)
    
    # Accuracy on testing features and label
    accuracy = accuracy_score(y_test, random_forest_predictions)
    print("Random Forest Testing Accuracy: {:.4f}%".format(accuracy * 100))
    
    # Calculate confusion matrix
    random_forest_conf_matrix = confusion_matrix(y_test, random_forest_predictions)
    print("Random Forest Confusion Matrix:")
    print(random_forest_conf_matrix)
    
    print("\n")
    
    # Calculate precision, recall, and F1-score
    random_forest_report = classification_report(y_test, random_forest_predictions)
    print("Random Forest Classification Report:")
    print(random_forest_report)
    
    print("Wait Tuning....\n")
    
    # Model Parameter Tuning
    # Define the parameter grid for Random Forest
    '''
    param_grid: This variable likely holds a dictionary specifying the grid of 
                parameters to search during the hyperparameter tuning process.
    'n_estimators': This parameter represents the number of trees in the forest.
                    In this case, the model is being tuned over three values:
                                      50, 100, and 200.
    'max_depth': This parameter controls the maximum depth of each tree in the 
                 forest. It's being tuned over 
                 three values: no maximum depth (None), 10, and 20.
                 
    The process takes a long time because it involves training multiple models
    with different combinations of these parameters to determine the optimal 
    configuration.
    
    there are a total of 3 (n_estimators) * 3 (max_depth) = 9 
    combinations of hyperparameters that the algorithm will evaluate.
    '''
    param_grid = {'n_estimators': [50, 100, 200], 'max_depth': [1, 10, 20]}
    
    # Perform grid search with cross-validation
    grid_search = GridSearchCV(RandomForestClassifier(), param_grid, cv=5)
    grid_search.fit(X_train, y_train)
    
    # Get the best classifier with tuned parameters
    best_random_forest = grid_search.best_estimator_
    
    # Predict the labels for the testing data using the best classifier
    best_random_forest_predictions = best_random_forest.predict(X_test)
    
    # Calculate accuracy on testing data using the best classifier
    accuracy_tuned = accuracy_score(y_test, best_random_forest_predictions)
    print("Tuned Random Forest Testing Accuracy: {:.4f}%".format(accuracy_tuned * 100))
    
    # Calculate confusion matrix using the best classifier
    confusion_tuned = confusion_matrix(y_test, best_random_forest_predictions)
    print("Tuned Random Forest Confusion Matrix:")
    print(confusion_tuned)
    
    print("\n")
    
    # Calculate precision, recall, and F1-score using the best classifier
    report_tuned = classification_report(y_test, best_random_forest_predictions)
    print("Tuned Random Forest Classification Report:")
    print(report_tuned)

In [11]:
random_forest()

Data loaded successfully:
         28.7967   16.0021  2.6449  0.3918  0.1982   27.7004    22.011  \
0       31.6036   11.7235  2.5185  0.5303  0.3773   26.2722   23.8238   
1      162.0520  136.0310  4.0612  0.0374  0.0187  116.7410  -64.8580   
2       23.8172    9.5728  2.3385  0.6147  0.3922   27.2107   -6.4633   
3       75.1362   30.9205  3.1611  0.3168  0.1832   -5.5277   28.5525   
4       51.6240   21.1502  2.9085  0.2420  0.1340   50.8761   43.1887   
...         ...       ...     ...     ...     ...       ...       ...   
19014   21.3846   10.9170  2.6161  0.5857  0.3934   15.2618   11.5245   
19015   28.9452    6.7020  2.2672  0.5351  0.2784   37.0816   13.1853   
19016   75.4455   47.5305  3.4483  0.1417  0.0549   -9.3561   41.0562   
19017  120.5135   76.9018  3.9939  0.0944  0.0683    5.8043  -93.5224   
19018  187.1814   53.0014  3.2093  0.2876  0.1539 -167.3125 -168.4558   

       -8.2027   40.092   81.8828  g  
0      -9.9574   6.3609  205.2610  g  
1     -45.2160  76